# 03 — Patch Dataset Construction

Extract H&E patches with Macenko stain normalization.

In [ ]:
from utils import st_helpers as st

ROOT, PHARMA = st.setup_pharma_paths()
st.set_seeds()
print('ROOT:', ROOT)
print('PHARMA:', PHARMA)


In [ ]:
from src.data import load_config, pharma_outputs_dir
from src.labels import build_labels_cohort
from src.patches import build_patch_cohort, fit_reference_stain, save_patch_index

cfg = load_config()
oncology = cfg['cohorts']['oncology']
all_slides = oncology + cfg['cohorts']['external'] + cfg['cohorts']['benchmark']
labels = build_labels_cohort(all_slides, cfg=cfg)
ref_stain = fit_reference_stain(oncology, cfg)
ref_stain


In [ ]:
build_patch_cohort(all_slides, ref_stain=ref_stain, cfg=cfg)

In [ ]:
idx_path = save_patch_index(labels)
print('Wrote', idx_path)
labels.groupby('slide_id').size()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from src.data import load_slide
from src.patches import extract_patch, coords_hires, patch_size_px, macenko_normalize, resize_patch
from utils import st_helpers as st

slide = oncology[0]
adata = load_slide(slide)
img = st.get_image(adata, 'hires')
_, half = patch_size_px(adata)
coords = coords_hires(adata)
fig, axes = plt.subplots(2, 6, figsize=(12, 4))
for ax, i in zip(axes.flat, np.linspace(0, len(coords)-1, 12, dtype=int)):
    x, y = coords[i]
    norm = macenko_normalize(extract_patch(img, x, y, half), ref_stain)
    ax.imshow(resize_patch(norm, 112)); ax.axis('off')
fig.savefig(pharma_outputs_dir() / 'stain_norm_montage.png', dpi=120, bbox_inches='tight')
plt.show()


**Next:** `04_train_cnn.ipynb`